> **2026** — local-first biology (Biopython/PDB/pandas); optional paid LLM only where noted. See `UPDATE_2026.md`.

# Chapter 7 — FASTA/GenBank Sequence QC (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2007.%20LangChain%20for%20Biology/LC4LSH_Chapter_7_FASTA_GenBank_Sequence_QC.ipynb)

**Learning objectives**
- Parse FASTA and GenBank records with Biopython
- Flag ambiguity codes and low-complexity regions
- Extract accession metadata and feature tables
- Produce a per-sequence QC report

> Runtime: ~5 min (local, no API)  
> Cost: free  
> Data: small built-in FASTA/GenBank-style records


## Environment setup


### Secrets (optional LLM only)


In [ ]:
import os
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except Exception:
    userdata = None
    IN_COLAB = False
if not IN_COLAB:
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass

def get_secret(name, default=None):
    if IN_COLAB and userdata is not None:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    return os.getenv(name, default)

# These notebooks run locally (Biopython/pandas); a paid LLM is OPTIONAL for narrative only.
API_KEY_PROVIDER = "OPENAI"  # "GEMINI" | "OPENAI" | "GROQ" | "ANTHROPIC"
if API_KEY_PROVIDER == "OPENAI":
    os.environ["OPENAI_API_KEY"] = get_secret("LC4LS_OPENAI_API_KEY", "sk-...")
print("Optional LLM provider:", API_KEY_PROVIDER, "(analysis runs without it)")
os.environ["HF_TOKEN"] = get_secret("HF_TOKEN", "") or ""


### Install pinned dependencies


In [ ]:
%pip install -q biopython "pandas>=2.0" "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4" "langchain==1.0.0" "langchain-openai==1.0.0" python-dotenv
# Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)


In [ ]:
LANGSMITH_API_KEY = get_secret("LANGSMITH_API_KEY", "")
LANGSMITH_PROJECT = "lc4lsh-chapter7-sequence-qc"
if LANGSMITH_API_KEY and LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ["LANGSMITH_TRACING"] = "true"
    os.environ["LANGSMITH_API_KEY"] = LANGSMITH_API_KEY
    os.environ["LANGSMITH_PROJECT"] = LANGSMITH_PROJECT
    print("LangSmith ON ->", LANGSMITH_PROJECT)
else:
    os.environ["LANGSMITH_TRACING"] = "false"
    print("LangSmith OFF (fine — these notebooks are local-first)")


## Why sequence QC?

Before any analysis, confirm each sequence is **what it claims to be**: valid alphabet, expected length, no excessive ambiguity or low-complexity artifact, and traceable accession metadata. QC prevents garbage-in/garbage-out downstream.


## 1. Parse FASTA records


In [ ]:
from io import StringIO
from Bio import SeqIO

FASTA = """>seq1 example kinase fragment
MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG
>seq2_with_ambiguity
MKTVRQERXKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGY
>seq3_low_complexity
AAAAAAAAAAAAAAAAAAKLLLLLLLLLLLLLLLLLL
"""
records = list(SeqIO.parse(StringIO(FASTA), "fasta"))
for r in records:
    print(r.id, len(r.seq), "residues")


## 2. Ambiguity & invalid-residue checks


In [ ]:
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

def ambiguity_report(seq):
    s = str(seq).upper()
    nonstd = sorted(set(s) - STANDARD_AA)
    frac = sum(1 for c in s if c not in STANDARD_AA) / max(len(s), 1)
    return {"nonstandard": nonstd, "nonstandard_frac": round(frac, 3)}

for r in records:
    rep = ambiguity_report(r.seq)
    flag = "FLAG" if rep["nonstandard_frac"] > 0 else "ok"
    print(r.id, rep, "->", flag)


## 3. Low-complexity detection


In [ ]:
from collections import Counter

def low_complexity_frac(seq, window=10, max_ident=8):
    s = str(seq)
    if len(s) < window:
        return 0.0
    low = 0
    for i in range(len(s) - window + 1):
        w = s[i:i+window]
        if Counter(w).most_common(1)[0][1] >= max_ident:
            low += 1
    return round(low / (len(s) - window + 1), 3)

for r in records:
    lc = low_complexity_frac(r.seq)
    print(r.id, "low_complexity_frac=", lc, "->", "FLAG" if lc > 0.3 else "ok")


## 4. Accession metadata & feature table (GenBank)


In [ ]:
GENBANK = """LOCUS       SEQ1        60 aa            linear   PRI 01-JAN-2000
DEFINITION  example kinase fragment.
ACCESSION   ABC12345
VERSION     ABC12345.1
FEATURES             Location/Qualifiers
     source          1..60
                     /organism="Homo sapiens"
     Domain          10..50
                     /note="kinase-like"
ORIGIN
        1 mktvrqerlk sivrilersk epvsgaqlae elsvsrqviv qdiaylrslg ynvatprgyv
//
"""
gb = next(SeqIO.parse(StringIO(GENBANK), "genbank"))
print("accession:", gb.name, "| len:", len(gb.seq))
for feat in gb.features:
    print(feat.type, feat.location, dict(feat.qualifiers))


## 5. Per-sequence QC summary table


In [ ]:
import pandas as pd

rows = []
for r in records:
    amb = ambiguity_report(r.seq)
    lc = low_complexity_frac(r.seq)
    rows.append({
        "id": r.id, "length": len(r.seq),
        "nonstandard_frac": amb["nonstandard_frac"],
        "low_complexity_frac": lc,
        "qc_pass": (amb["nonstandard_frac"] == 0 and lc <= 0.3),
    })
qc = pd.DataFrame(rows)
print(qc.to_string(index=False))
print("\nPassing:", int(qc["qc_pass"].sum()), "/", len(qc))


## Limitations & safety notes

- QC flags are heuristics (thresholds on ambiguity/low-complexity); tune per use case.
- A passing sequence is not evidence of function — QC only checks sequence sanity.
- GenBank parsing here is minimal; real records have richer feature/ontology structure.
- Local/free; no API needed.


In [ ]:
# Cleanup
import gc
for _v in ("records", "df", "model", "llm", "structure", "X"):
    globals().pop(_v, None)
gc.collect()
print("Cleanup complete.")


## Exercises

<details><summary>Why flag ambiguity codes (X, N, etc.)?</summary>They mark uncertain bases/residues; high rates can break alignments, translations, and downstream models.</details>

<details><summary>Why detect low-complexity regions?</summary>Homopolymer/repeat stretches are often artifacts or special cases that skew statistics and similarity searches.</details>

<details><summary>Why keep accession metadata?</summary>Accessions make a sequence traceable to a curated database entry and version.</details>

### Tasks
- **Task A** - Add a length-distribution plot and flag outliers beyond 2 std from the mean.
- **Task B** - Add translation-frame QC for nucleotide CDS records (internal stop codons).
- **Task C** - Parse a real GenBank file and summarize feature-type counts.
- **Task D** - Export the QC table to CSV with a JSON manifest of thresholds used.
